In [3]:
# Install
!pip install openai pypdf sentence-transformers faiss-cpu gradio

# Imports
from openai import OpenAI
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
import gradio as gr
import os

# 🔑 Set API key (for Colab)
os.environ["OPENAI_API_KEY"] = "YOUR_OPENAI_API_KEY"
client = OpenAI()

# Globals
chunks = []
index = None
embed_model = SentenceTransformer('all-MiniLM-L6-v2')

# -------------------------
# Process PDF
# -------------------------
def process_pdf(file):
    global chunks, index

    if file is None:
        return "❌ Upload PDF first"

    reader = PdfReader(file.name)
    text = ""

    for page in reader.pages:
        t = page.extract_text()
        if t:
            text += t

    if not text.strip():
        return "❌ No readable text"

    chunks = [text[i:i+500] for i in range(0, len(text), 500)]

    embeddings = embed_model.encode(chunks)
    index = faiss.IndexFlatL2(len(embeddings[0]))
    index.add(np.array(embeddings))

    return f"✅ PDF processed ({len(chunks)} chunks)"

# -------------------------
# Search
# -------------------------
def search(query, k=3):
    query_vec = embed_model.encode([query])
    D, I = index.search(np.array(query_vec), k)
    return " ".join([chunks[i] for i in I[0]])

# -------------------------
# Ask with MODE
# -------------------------
def ask(query, mode):
    if index is None:
        return "❌ Upload PDF first"

    context = search(query)[:1500]

    system_prompts = {
        "Beginner": "Explain in simple terms with examples.",
        "Exam": "Answer in structured format with bullet points suitable for exams.",
        "Strict": "Answer ONLY from the given context. If not found, say 'Not in document'."
    }

    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": system_prompts[mode]},
            {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {query}"}
        ]
    )

    return response.choices[0].message.content

# -------------------------
# Generate Questions
# -------------------------
def generate_questions():
    if not chunks:
        return "❌ Upload PDF first"

    context = " ".join(chunks[:5])[:1500]

    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": "You are an exam paper setter."},
            {"role": "user", "content": f"""
            Based on this:
            {context}

            Generate:
            - 3 short questions
            - 3 MCQs with answers
            - 2 long questions
            """}
        ]
    )

    return response.choices[0].message.content

# -------------------------
# Summarize
# -------------------------
def summarize():
    if not chunks:
        return "❌ Upload PDF first"

    context = " ".join(chunks[:5])[:1500]

    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": "Summarize in bullet points."},
            {"role": "user", "content": context}
        ]
    )

    return response.choices[0].message.content

# -------------------------
# UI (ContextIQ)
# -------------------------
with gr.Blocks(theme=gr.themes.Soft(), title="ContextIQ") as app:

    gr.Markdown("""
    # 🧠 ContextIQ
    ### AI Document Intelligence Assistant
    """)

    file_input = gr.File(label="📂 Upload PDF")
    upload_btn = gr.Button("⚙️ Process PDF")
    status = gr.Textbox(label="Status")

    upload_btn.click(process_pdf, inputs=file_input, outputs=status)

    gr.Markdown("## 💬 Ask Questions")

    mode = gr.Dropdown(
        ["Beginner", "Exam", "Strict"],
        value="Beginner",
        label="Select Mode"
    )

    user_input = gr.Textbox(label="Your Question")
    ask_btn = gr.Button("Ask")

    answer_output = gr.Textbox(label="Answer", lines=8)

    ask_btn.click(ask, inputs=[user_input, mode], outputs=answer_output)

    gr.Markdown("## 🧠 Smart Features")

    q_btn = gr.Button("📝 Generate Questions")
    s_btn = gr.Button("📌 Summarize")

    extra_output = gr.Textbox(label="Output", lines=12)

    q_btn.click(generate_questions, outputs=extra_output)
    s_btn.click(summarize, outputs=extra_output)

app.launch()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/tmp/ipykernel_31778/4169767470.py:132: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), title="ContextIQ") as app:


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://4640c7c4e42c66c58d.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
